# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Scope note

This notebook uses `work/model_df_export.csv` (exported from `w03_data_contract`'s
`feature_df` + `honest_proxy_df`, 331,437 rows, one row per unique `content_hash_id` —
verified with `duplicated()`, zero duplicates) merged with `work/baseline_action_score.csv`
for the Week-4 baseline comparison. Feature set: `march_impressions`, `avg_search_position`,
`march_sessions`, `engagement_rate`, `march_scroll_events`. Target: `is_declining_proxy`
(April clicks < March clicks — built with proper time separation upstream in
`w03_data_contract`). Note `march_clicks` is correctly absent from this export — it was
excluded as the deliberate leakage trap in Week 3 and stays excluded here.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import pandas as pd
import numpy as np

model_df = pd.read_csv("work/model_df_export.csv")
baseline = pd.read_csv("work/baseline_action_score.csv")[["content_hash_id", "baseline_score"]]

df = model_df.merge(baseline, on="content_hash_id", how="inner")
print("Merged rows:", len(df), "/ original model_df rows:", len(model_df))

feature_cols = ["march_impressions", "avg_search_position", "march_sessions",
                "engagement_rate", "march_scroll_events"]
print("\nMissing values per feature:")
print(df[feature_cols].isna().sum())
print("\nTarget balance:")
print(df["is_declining_proxy"].value_counts(normalize=True).round(4))


Merged rows: 331437 / original model_df rows: 331437

Missing values per feature:
march_impressions           0
avg_search_position    154699
march_sessions              0
engagement_rate        241200
march_scroll_events         0
dtype: int64

Target balance:
is_declining_proxy
0    0.8639
1    0.1361
Name: proportion, dtype: float64


**Method choice: Random Forest classifier.** The Week 4 baseline is a hand-written linear
rule over two signals. The lane's own reasoning in `w02_ml_task_framing` (Section 5) is that
ML should earn its place by combining *multiple* signals rather than one threshold — a
Random Forest fits that directly: it can pick up non-linear interactions between
`march_impressions`, `avg_search_position`, `march_sessions`, `engagement_rate`, and
`march_scroll_events` that a single weighted-sum rule cannot, while staying easy to inspect
via feature importances for the error analysis in Section 4. `is_declining_proxy` is
imbalanced (~13.6% positive), so `class_weight="balanced"` is used rather than treating the
classes as equal by default.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].copy()
y = df["is_declining_proxy"].copy()

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=y
)

print("Train rows:", len(train_idx), " Test rows:", len(test_idx))
print("Train positive rate:", y.loc[train_idx].mean().round(4))
print("Test positive rate:", y.loc[test_idx].mean().round(4))


Train rows: 265149  Test rows: 66288
Train positive rate: 0.1361
Test positive rate: 0.1361


**Split design: random, stratified, at the content-page level — not grouped, not
time-aware, and here's why that's the honest choice rather than an oversight.**
`content_hash_id` is unique in this export (verified above, zero duplicates), so there is no
"same page appears in both train and test" leakage risk that a grouped-by-client split would
normally guard against — this export doesn't even carry a `client_hash_id` to group by.
Time-awareness was already handled one level upstream, in `w03_data_contract`: every feature
here is aggregated from March 2026 only, and the label (`is_declining_proxy`) compares April
clicks to March clicks — the future-vs-past separation that matters is baked into the data
construction, not something a row-level split needs to re-enforce. What a plain random split
does need to guard is stratification on the imbalanced target, which is applied above.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X.loc[train_idx])
X_test = imputer.transform(X.loc[test_idx])
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-scores)[:k]
    return labels.to_numpy()[order].mean()

# Model: rank the TEST set by predicted probability
model_p50 = precision_at_k(y_test, test_scores, k=50)

# Week-4 baseline: same TEST rows, ranked by baseline_score instead — same data, same metric
baseline_test_scores = df.loc[test_idx, "baseline_score"].to_numpy()
baseline_p50 = precision_at_k(y_test, baseline_test_scores, k=50)

comparison = pd.DataFrame([
    {"model": "Week-4 baseline (baseline_score)", "Precision@50": round(baseline_p50, 4)},
    {"model": "Week-5 Random Forest", "Precision@50": round(model_p50, 4)},
])
comparison


,model,Precision@50
0,Week-4 baseline (baseline_score),0.42
1,Week-5 Random Forest,0.66


**Same test rows, same metric (Precision@50 from `w02_ml_task_framing`), same 80/20 split
— only the ranking score changes between the two rows above.** The baseline was never built
to predict `is_declining_proxy`, so this is an honest, if slightly unfair to the baseline,
comparison: the Random Forest gets to see labeled training data aligned with exactly the
metric it's judged on, while the baseline rule was designed by hand against a different
notion of "opportunity." The comparison still answers the practical question the paper needs:
given the same 66,288 held-out pages, which ranking puts more truly-declining pages in the
top 50?

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances.round(4))
print()

# Look at the top 50 by MODEL score within the test set, and see how many are false positives
order = np.argsort(-test_scores)[:50]
top50 = df.loc[test_idx].iloc[order].copy()
top50["predicted_prob"] = test_scores[order]
top50["correct"] = top50["is_declining_proxy"] == 1

print("Top-50 by model score — correct vs false positive:")
print(top50["correct"].value_counts())
print()
print("Feature averages, correct hits vs false positives (top 50):")
print(top50.groupby("correct")[feature_cols].mean().round(2))


Feature importances:
march_impressions      0.6320
march_sessions         0.1709
avg_search_position    0.1695
march_scroll_events    0.0220
engagement_rate        0.0056
dtype: float64

Top-50 by model score — correct vs false positive:
correct
True     33
False    17
Name: count, dtype: int64

Feature averages, correct hits vs false positives (top 50):
         march_impressions  avg_search_position  march_sessions  \
correct                                                           
False             11003.71                 5.51           77.41   
True              11874.30                 5.52           66.91   

         engagement_rate  march_scroll_events  
correct                                        
False               0.07                14.29  
True                0.08                11.21  


**Where the model is wrong:** feature importances put `march_impressions` far in front (0.632), with `march_sessions` and `avg_search_position` a distant second and third (0.171, 0.170), and `engagement_rate` barely used (0.006). In the top 50 by model score within the test set, 33 of 50 are correct hits and 17 are false positives. The false positives are **not** distinguishable from the true positives by impressions or position — both groups average ~11,000-11,900 impressions and ~5.5 average position — but the false positives have somewhat *higher* `march_sessions` (77.4 vs 66.9) and `march_scroll_events` (14.3 vs 11.2) than the true positives. Read plainly: among the highest-impression pages, the model is not yet distinguishing well between a page that is actually declining and one that carries similar traffic and position but slightly deeper session engagement. That is a concrete, specific error pattern for the paper's Limitations section, not a generic "the model has some false positives" caveat.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.